# Experiment 5 - Credit Card Fraud Detection
**Date:** 04/08/2026 &emsp; **Reg No:** 25BDS0151

**Objective:** Detect outliers for identifying fraudulent transactions in a credit card transactions dataset using Discretization & Binning, Feature Scaling & Normalization, and Outlier Detection techniques.

**Dataset:** `fraudTrain.csv` and `fraudTest.csv`

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
# Load the datasets
df_train = pd.read_csv('fraudTrain.csv')
df_test = pd.read_csv('fraudTest.csv')

print('Training set shape:', df_train.shape)
print('Test set shape:', df_test.shape)

In [ ]:
# Combine both datasets for analysis
df = pd.concat([df_train, df_test], ignore_index=True)
print('Combined dataset shape:', df.shape)
df.head()

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
# Check fraud distribution
print('Fraud distribution:')
print(df['is_fraud'].value_counts())
print('\nFraud percentage: {:.2f}%'.format(df['is_fraud'].mean() * 100))

---
## Exp 5.1 - Discretization and Binning

### 5.1.1 - Using pd.cut() with custom bins on Transaction Amount

In [ ]:
# Define custom bins for transaction amounts
bins = [0, 10, 50, 100, 500, 1500, 30000]
category = pd.cut(df['amt'], bins)
category

In [ ]:
# Count the number of values in each bin
pd.value_counts(category)

In [ ]:
# We can also indicate bin names by passing a list of labels
bin_names = ['Very Low', 'Low', 'Medium', 'High', 'Very High', 'Extreme']
category_labeled = pd.cut(df['amt'], bins, labels=bin_names)
category_labeled

In [ ]:
pd.value_counts(category_labeled)

### 5.1.2 - Using pd.cut() with integer bins (equal-width)

In [ ]:
# If we pass just an integer for bins, it computes equal-length bins
# based on the minimum and maximum values in the data
category2 = pd.cut(df['amt'], 5, precision=2)
category2

In [ ]:
pd.value_counts(category2)

### 5.1.3 - Using pd.qcut() (Quantile-based binning)

In [ ]:
# pd.qcut forms bins based on sample quantiles
# cut into quartiles
category3 = pd.qcut(df['amt'], 4)
category3

In [ ]:
# Based on the number of bins (4), it converted our data into four different categories
# If we count the number of values in each category, we should get equal-sized bins
pd.value_counts(category3)

### 5.1.4 - Binning on city_pop with labels

In [ ]:
# Apply binning on city population
pop_bins = [0, 5000, 25000, 100000, 500000, 2000000]
pop_labels = ['Rural', 'Small Town', 'Medium City', 'Large City', 'Metro']
pop_category = pd.cut(df['city_pop'], pop_bins, labels=pop_labels)
pop_category

In [ ]:
pd.value_counts(pop_category)

---
## Exp 5.2 - Feature Scaling and Normalization

In [ ]:
from sklearn.preprocessing import MinMaxScaler, StandardScaler

In [ ]:
# Select numerical columns for scaling
data = df[['amt', 'lat', 'long', 'city_pop', 'merch_lat', 'merch_long']].copy()
print("Original Data:")
data.head(10)

In [ ]:
data.describe()

### 5.2.1 - Min-Max Scaling (Normalization)
Rescales features to a fixed range, typically 0 to 1.

In [ ]:
# Min-Max Scaling (Normalization)
# Rescales features to a fixed range, typically 0 to 1
min_max_scaler = MinMaxScaler()
df_min_max = pd.DataFrame(
    min_max_scaler.fit_transform(data),
    columns=data.columns
)
print("Min-Max Scaled Data:")
df_min_max.head(10)

In [ ]:
df_min_max.describe()

### 5.2.2 - Z-Score Standardization
Centers data around mean 0 with a standard deviation of 1.

In [ ]:
# Z-score Standardization
# Centers data around mean 0 with a standard dev of 1
standard_scaler = StandardScaler()
df_standard = pd.DataFrame(
    standard_scaler.fit_transform(data),
    columns=data.columns
)
print("Standardized Data:")
df_standard.head(10)

In [ ]:
df_standard.describe()

---
## Exp 5.3 - Outlier Detection

In [ ]:
# First, let's look at the transaction amount statistics
print('Transaction Amount Statistics:')
print(df['amt'].describe())

### 5.3.1 - Detecting outliers using np.abs() and threshold filtering

In [ ]:
# Let's find transactions where the amount exceeds 500
TotalTransaction = df['amt']
TotalTransaction[np.abs(TotalTransaction) > 500]

In [ ]:
# Count of outlier transactions
outliers = TotalTransaction[np.abs(TotalTransaction) > 500]
print(f'Number of transactions exceeding $500: {len(outliers)}')
print(f'Total transactions: {len(df)}')
print(f'Percentage: {len(outliers)/len(df)*100:.2f}%')

In [ ]:
# Display all columns and rows where amt is greater than 500
df[np.abs(TotalTransaction) > 500]

In [ ]:
# Check the fraud rate in these outlier transactions
outlier_rows = df[np.abs(TotalTransaction) > 500]
normal_rows = df[np.abs(TotalTransaction) <= 500]

print(f'Fraud rate in outlier transactions (amt > $500): {outlier_rows["is_fraud"].mean()*100:.2f}%')
print(f'Fraud rate in normal transactions (amt <= $500): {normal_rows["is_fraud"].mean()*100:.2f}%')

### 5.3.2 - Detecting outliers at a higher threshold

In [ ]:
# Let's find transactions where amount exceeds 1000
TotalTransaction[np.abs(TotalTransaction) > 1000]

In [ ]:
# Display all columns for these high-value outlier transactions
df[np.abs(TotalTransaction) > 1000]

In [ ]:
# Fraud analysis on high-value outliers
high_outliers = df[np.abs(TotalTransaction) > 1000]
print(f'Number of transactions exceeding $1000: {len(high_outliers)}')
print(f'Fraud rate in these transactions: {high_outliers["is_fraud"].mean()*100:.2f}%')

### 5.3.3 - Outlier detection on city_pop

In [ ]:
# Detect outliers in city_pop
CityPop = df['city_pop']
print('City Population Statistics:')
print(CityPop.describe())

In [ ]:
# Find cities with population exceeding 500000
CityPop[np.abs(CityPop) > 500000]

In [ ]:
# Display full details of transactions in high-population cities
df[np.abs(CityPop) > 500000]

In [ ]:
# Fraud rate in high-population city transactions vs others
high_pop = df[np.abs(CityPop) > 500000]
low_pop = df[np.abs(CityPop) <= 500000]

print(f'Transactions in cities with pop > 500000: {len(high_pop)}')
print(f'Fraud rate in high-pop cities: {high_pop["is_fraud"].mean()*100:.2f}%')
print(f'Fraud rate in other cities: {low_pop["is_fraud"].mean()*100:.2f}%')

---
## Conclusion

1. **Discretization & Binning:** Using `pd.cut()` and `pd.qcut()`, we converted continuous features like transaction amount and city population into discrete bins. Custom bins with labels and equal-width/quantile bins were explored.

2. **Feature Scaling & Normalization:** Min-Max Scaling and Z-Score Standardization were applied to numerical features. Min-Max scales values to [0,1] while Z-Score centers data around mean 0 with standard deviation 1.

3. **Outlier Detection:** Using `np.abs()` with threshold filtering, outlier transactions were identified. Transactions exceeding $500 and $1000 showed significantly higher fraud rates compared to normal transactions, demonstrating that outlier detection is effective for fraud identification.